In [1]:
import sys
sys.path.append("..")

In [2]:
from src import load_dataset
from src.preprocessing import show_missing_entries
import src.feature_engineering as feat
from src.imputation import ImputeByGroup, FillCabin, FillNA
from src.log import save_load_model_hist

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression

# Notebook Configs

In [3]:
SEED = 42
DEBUGGING_MODE = True
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [4]:
train_set, test_set = load_dataset()

# Pipeline

## Imputer

In [5]:
imputer_pipeline = Pipeline([
    ('impute_fare', ImputeByGroup(group_by_col = ['Pclass'], col_to_impute = 'Fare')),
    ('impute_age', ImputeByGroup(group_by_col = ['Pclass', 'Sex'], col_to_impute = 'Age', strategy = 'median')),
    ('impute_cabin', FillCabin()),
    ('impute_cabin_embarked', FillNA())
])

## Feature Engineering

In [6]:
feature_pipeline = Pipeline([
    ("Family_size", feat.FeatFamilySize()),
    ("Is_solo", feat.FeatIsSolo()),
    ("Fare_bucket", feat.FeatFareBucket()),
    ("Age_bucket", feat.FeatAgeBucket()),
    ("Deck", feat.FeatDeck()),
    ("Deck_missing", feat.FeatDeckMissing()),
    ("Title", feat.FeatTitle()),
    ("Is_ticket_num", feat.FeatIsTicketNum()),
    ("Ticket_char_0", feat.FeatTicketFirstChar()),
])

## Drop UNENCODED columns

In [7]:
cols_to_drop = ['PassengerId',
                'Name', 
                'Ticket', 
                'Cabin', 
                'SibSp', 
                'Parch'
               ]

drop_pipeline = Pipeline([
    ('drop', feat.DropColumns(cols_to_drop = cols_to_drop)) 
])

## Encode

In [8]:
col_type_to_encode = ['object', 'category', 'str']
one_hot_encode = (ColumnTransformer([
    (
        'hot_encode', 
         OneHotEncoder(sparse_output=False, handle_unknown='ignore'), 
         make_column_selector(dtype_include=col_type_to_encode)
    )
], remainder = "passthrough", verbose_feature_names_out=False)).set_output(transform='pandas')

## Drop ENCODED columns

### TODO

## Orchestration Pipeline

In [9]:
final_preprocess_pipeline = Pipeline([
    ('impute', imputer_pipeline),
    ('feature', feature_pipeline),
    ('drop_unencoded', drop_pipeline),
    ('encode', one_hot_encode)
])

In [10]:
no_encode_preprocess_pipeline = Pipeline([
    ('impute', imputer_pipeline),
    ('feature', feature_pipeline),
    ('drop_unencoded', drop_pipeline)
])

## Common Description

In [24]:
df = train_set.copy()
transformed_df = no_encode_preprocess_pipeline.fit_transform(df)
final_cols = no_encode_preprocess_pipeline.get_feature_names_out()
cols_to_encode = transformed_df.select_dtypes(include=col_type_to_encode).columns.to_list()
impute_desc = no_encode_preprocess_pipeline.named_steps['impute']

In [31]:
desc = f"""Impute:
{impute_desc}

Final Transformed Features:
{final_cols}

Encoded Columns:
{cols_to_encode}

Scaler:
StandardScaler() for non-tree models
"""

In [32]:
print(desc)

Impute:
Pipeline(steps=[('impute_fare',
                 ImputeByGroup(col_to_impute='Fare', group_by_col=['Pclass'])),
                ('impute_age',
                 ImputeByGroup(col_to_impute='Age',
                               group_by_col=['Pclass', 'Sex'],
                               strategy='median')),
                ('impute_cabin', FillCabin()),
                ('impute_cabin_embarked', FillNA())])

Final Transformed Features:
['Survived' 'Pclass' 'Sex' 'Age' 'Fare' 'Embarked' 'Family_size' 'Is_solo'
 'Fare_bucket' 'Age_bucket' 'Deck' 'Deck_missing' 'Title' 'Is_ticket_num'
 'Ticket_char_0']

Encoded Columns:
['Sex', 'Embarked', 'Fare_bucket', 'Age_bucket', 'Deck', 'Title', 'Ticket_char_0']

Scaler:
StandardScaler() for non-tree models



# Modeling

In [ ]:
df_train = train_set.copy()
X = df_train.drop(labels = ['Survived'], axis = 1)
y = df_train['Survived']

## Logistic Regression

In [ ]:
lr_model = Pipeline([
    ('preproces', final_preprocess_pipeline),
    ('scale', StandardScaler().set_output(transform="pandas")),
    ('predictor', LogisticRegression(max_iter = 1000, random_state = SEED, C = 0.1, solver ='lbfgs'))
])

scores = cross_val_score(lr_model, X, y, cv=skf)
print(scores.mean(), scores.std())
latest, last_algo_stat, model_hist = save_load_model_hist('LR', cv_scores=scores, description=desc, clear_model_history = False)
print("Latest")
print(latest[['Model', 'Mean', 'Std']])
print("Former")
print(last_algo_stat[['Model', 'Mean', 'Std']])
model_hist[['Model', 'Mean', 'Std']].sort_values(by='Mean', ascending=False).head(5)